# Overview to Convolutional Neural Networks (CNNs)
## Image Classification, Hyperparameter Tuning & Regularization with PyTorch


---

## Contents

1. [Introduction & Learning Objectives](#introduction)
2. [Prerequisites & Imports](#imports)
3. [Part 1: Quick Tutorial – CNN on MNIST](#part1)
   - [3.1 Load and Visualize MNIST](#31)
   - [3.2 Define a Simple CNN](#32)
   - [3.3 Training Setup](#33)
   - [3.4 Training Loop + Live Plot](#34)
   - [3.5 Evaluation & Confusion Matrix](#35)
   - [3.6 Quick Hyperparameter Experiment](#36)
4. [Part 2: Student Tasks & grading](#part2)
5. [Lab Summary & Takeaways](#summary)
6. [References & Further Reading](#refs)

---

<a id="introduction"></a>
## Overview

Convolutional Neural Networks (**CNNs**) learn **spatial hierarchies of features** from images using **convolution**, **nonlinearities**, and **pooling**. They are the standard backbone for image understanding and underpin many **generative** systems (e.g., encoders in **VAEs**, discriminators in **GANs**, and **U-Net**-style architectures in **diffusion** models).

### Goals

By the end of this lab, you will be able to:

- **Load** image data with `torchvision` and **batch** it with `DataLoader`.
- **Build** a small CNN in **PyTorch** (`nn.Module`, conv/pool/FC layers).
- **Train** a classifier with a standard **loss**, **optimizer**, and **GPU/CPU** device placement.
- **Interpret** training curves and a **confusion matrix**.
- **Experiment** with **hyperparameters**, **dropout**, **weight decay**, and **data augmentation**.

### Reading this notebook

- Run cells **in order** the first time.
- Short explanations appear **before** major code cells; read them before running.
- **Part 2** is graded / self-checked—complete tasks in order when instructed.


<a id="imports"></a>
## Prerequisites & Imports

You need **Python 3** with **PyTorch**, **torchvision**, and plotting libraries. This cell prints versions for reproducibility and selects **CPU or CUDA**.


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.metrics import confusion_matrix
import seaborn as sns

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

<a id="part1"></a>
## Part 1: Quick Tutorial – Training a CNN on MNIST (≈40–45 minutes)

**MNIST** is 28×28 grayscale digits (10 classes). It trains quickly—ideal for learning CNN mechanics before scaling to harder datasets or generative models.


<a id="31"></a>
### 3.1 Load and Visualize MNIST (≈8 min)

We use **`transforms`** to convert images to tensors and **normalize** pixel values. **`DataLoader`** shuffles training batches and can use multiple workers for faster loading.


In [ ]:
mnist_transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
data_dir = os.path.join(os.getcwd(), "data")
train_ds = torchvision.datasets.MNIST(root=data_dir, train=True, download=True, transform=mnist_transform)
test_ds = torchvision.datasets.MNIST(root=data_dir, train=False, download=True, transform=mnist_transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

<a id="32"></a>
### 3.2 Define a Simple CNN Architecture (≈10 min)

**Architecture sketch:** three **Conv → ReLU → MaxPool** blocks extract hierarchical features; **fully connected** layers map features to 10 digit classes.

| Layer | Role |
|--------|------|
| `Conv2d` | Local feature detectors (edges, strokes) |
| `ReLU` | Nonlinearity |
| `MaxPool2d` | Spatial downsampling + translation robustness |
| `Linear` | Class scores |


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

<a id="33"></a>
### 3.3 Training Setup (≈5 min)

We use **cross-entropy** (multi-class classification) and **Adam** with learning rate **0.001**. Hyperparameters: **batch size 64**, **5 epochs** (enough for a strong baseline on MNIST).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LR = 1e-3
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.0  # tutorial baseline; Part 2 explores L2 regularization

net = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f"Loss: CrossEntropyLoss | Optimizer: Adam(lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"Epochs: {NUM_EPOCHS} | Batch size: {BATCH_SIZE} | Device: {device}")


<a id="34"></a>
### 3.4 Training Loop + Live Plot (≈12 min)

Each epoch we compute **training loss/accuracy** and **test loss/accuracy**. The plot **refreshes every epoch** so you can watch learning progress. `%%time` reports wall-clock time for the whole training block.


In [ ]:
%%time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

net = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def run_epoch(net, loader, train: bool):
    if train:
        net.train()
    else:
        net.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            logits = net(xb)
            loss = criterion(logits, yb)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(net, train_loader, train=True)
    te_loss, te_acc = run_epoch(net, test_loader, train=False)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss)
    history["test_acc"].append(te_acc)

    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    epochs_x = range(1, len(history["train_loss"]) + 1)
    ax[0].plot(epochs_x, history["train_loss"], "o-", label="Train")
    ax[0].plot(epochs_x, history["test_loss"], "s-", label="Test")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Loss")
    ax[0].set_title("Loss per epoch")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(epochs_x, history["train_acc"], "o-", label="Train")
    ax[1].plot(epochs_x, history["test_acc"], "s-", label="Test")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Accuracy")
    ax[1].set_title("Accuracy per epoch")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    plt.suptitle(f"Epoch {epoch}/{NUM_EPOCHS} — Train acc: {tr_acc:.4f} | Test acc: {te_acc:.4f}", fontweight="bold")
    plt.tight_layout()
    plt.show()

print("Final — Train acc:", history["train_acc"][-1], "Test acc:", history["test_acc"][-1])


<a id="35"></a>
### 3.5 Final Evaluation & Confusion Matrix (≈5 min)

A **confusion matrix** shows which digits are confused with which—useful for error analysis before moving to harder datasets or generative evaluation.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def collect_predictions(net, loader):
    net.eval()
    ys, yhats = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = net(xb)
        pred = logits.argmax(dim=1).cpu().numpy()
        ys.append(yb.numpy())
        yhats.append(pred)
    return np.concatenate(ys), np.concatenate(yhats)


y_true, y_pred = collect_predictions(net, test_loader)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("MNIST Test — Confusion Matrix")
plt.tight_layout()
plt.show()


<a id="36"></a>
### 3.6 Quick Hyperparameter Experiment (≈5 min)

Compare **two learning rates** (e.g. **0.01** vs **0.001**) with the same seed and epochs. **Expectation:** too large a learning rate may oscillate or underperform; too small may learn slowly.


In [ ]:
%%time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_mnist_cnn(lr: float, epochs: int = 5, seed: int = 123):
    """Self-contained train loop (does not rely on run_epoch from §3.4)."""
    set_seed(seed)
    m = SimpleCNN(10).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "test_acc": []}
    for _ in range(epochs):
        m.train()
        running_loss, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(m(xb), yb)
            loss.backward()
            opt.step()
            running_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        tr_loss = running_loss / max(n, 1)
        m.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = m(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        te_acc = correct / max(total, 1)
        hist["train_loss"].append(tr_loss)
        hist["test_acc"].append(te_acc)
    return m, hist


lr_a, lr_b = 0.01, 0.001
_, h_high = train_mnist_cnn(lr_a, epochs=5, seed=123)
_, h_low = train_mnist_cnn(lr_b, epochs=5, seed=123)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), h_high["test_acc"], "o-", label=f"Test acc | lr={lr_a}")
plt.plot(range(1, 6), h_low["test_acc"], "s-", label=f"Test acc | lr={lr_b}")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Quick comparison: learning rate vs test accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final test acc (lr={lr_a}): {h_high['test_acc'][-1]:.4f}")
print(f"Final test acc (lr={lr_b}): {h_low['test_acc'][-1]:.4f}")


<a id="part2"></a>
## Part 2: Hands-On Student Tasks (≈85–95 minutes)

**Instructions:** Complete the tasks below in order. Use tables/plots where asked. Keep **batch size**, **epochs**, and **seeds** **documented** in your write-up so results are comparable. **Do not copy full solutions from the web**—build your own code in the cells provided.

> **Tip:** Copy helper functions (`run_epoch`, net classes) from Part 1 into your solution cells or refactor into a single code cell at the top of Part 2.

When a task asks for **analysis**, **tables**, or **figures**, add **Markdown cells** below your code and paste plots or type your answers there.

<a id="grading"></a>
### Grading rubric (100 points total)

| Task | Topic | Points |
|------|--------|--------:|
| **Task 1** | Hyperparameter tuning (≥3 configs, plot + analysis) | **20** |
| **Task 2** | Dropout regularization (train vs test, discussion) | **20** |
| **Task 3** | L2 / weight decay (curves + interpretation) | **20** |
| **Task 4** | Data augmentation (baseline comparison) | **20** |
| **Task 5** | Deeper CNN architecture (comparison vs `SimpleCNN`) | **20** |
| | **Total** | **100** |

*Optional **Bonus** task below is **not** part of the 100 points; instructors may award **extra credit** separately.*


### Part 2 Setup: Shared Helpers and Baseline Model
To solve the tasks below, we first ensure the `SimpleCNN` and `train_eval` helper are defined and available.

In [ ]:
# Task 1: Hyperparameter Tuning, Setup & Data Loading
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

# Define global hyperparams used in print statements
BATCH_SIZE = 64

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_eval(net, t_loader, lr=1e-3, wd=0.0, opt_name="Adam", epochs=3):
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd) if opt_name=="Adam" else torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        for xb, yb in t_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(net(xb), yb).backward()
            opt.step()
    net.eval()
    c, t = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (net(xb).argmax(1)==yb).sum().item()
            t += yb.size(0)
    return c/t

data_dir = os.path.join(os.getcwd(), "data")
mnist_transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train_ds = torchvision.datasets.MNIST(root=data_dir, train=True, download=True, transform=mnist_transform)
test_ds = torchvision.datasets.MNIST(root=data_dir, train=False, download=True, transform=mnist_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

results = []
for cfg in [{"n":"Adam_1e-3","lr":1e-3,"opt":"Adam"}, {"n":"Adam_1e-2","lr":1e-2,"opt":"Adam"}, {"n":"SGD_1e-2","lr":1e-2,"opt":"SGD"}]:
    set_seed(42)
    m = SimpleCNN().to(device)
    acc = train_eval(m, train_loader, lr=cfg["lr"], opt_name=cfg["opt"])
    results.append((cfg["n"], acc))

n, a = zip(*results)
plt.bar(n, a)
plt.title("Task 1: Hparams Comparison")
plt.ylabel("Accuracy")
plt.show()

### Task 2 (≈20 min): Add Dropout Regularization — **20 / 100 points**

**Goal:** Insert **`nn.Dropout(p)`** after activation in each conv block and **before** fully-connected layers (try **p = 0.25** and **0.5**). Retrain with the same hyperparameters as the tutorial baseline. Compare **train vs test curves** to discuss **overfitting**.

**Deliverable:** Plot loss/accuracy for baseline vs dropout net(s) + short explanation.

**Hints:**
- Subclass `nn.Module` or duplicate **`SimpleCNN`** from Part 1 and add `nn.Dropout` layers.
- **`net.train()`** enables dropout; **`net.eval()`** disables it—use `eval()` when measuring test accuracy.
- You may see **higher train loss** with dropout but **closer train–test gap** (less overfitting).


In [ ]:
# Task 2: Dropout Regularization
class SimpleCNNWithDropout(nn.Module):
    def __init__(self, p=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(p)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(p)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.drop3 = nn.Dropout(p)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.drop1(self.pool1(F.relu(self.conv1(x))))
        x = self.drop2(self.pool2(F.relu(self.conv2(x))))
        x = self.pool3(F.relu(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.drop3(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

set_seed(42)
m_drop = SimpleCNNWithDropout(p=0.25).to(device)
acc = train_eval(m_drop, train_loader)
print(f"Task 2 Dropout (p=0.25) Acc: {acc:.4f}")

### Task 3 (≈15 min): L2 Regularization (Weight Decay) — **20 / 100 points**

**Goal:** Add **`weight_decay`** (e.g. **1e-4** or **5e-4**) to **Adam** (or SGD). Compare **training and test loss curves** against **weight_decay = 0**.

**Deliverable:** Overlayed loss curves + 2–3 sentences on whether test loss improved.

**Hints:**
- Pass `weight_decay=...` into **`torch.optim.Adam`** (or SGD)—see [PyTorch Adam docs](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html).
- Run the **same** number of epochs for each setting; overlay **train** and **test** loss on one figure (different line styles/colors).
- If curves are noisy, fix the seed or average over a short window—focus on **trend**.


In [ ]:
# Task 3: L2 Regularization (Weight Decay)
wd_results = {}
for wd in [0.0, 1e-4, 5e-4]:
    print(f"Training with weight_decay={wd}...")
    set_seed(42)
    m = SimpleCNN().to(device)
    acc = train_eval(m, train_loader, wd=wd)
    wd_results[str(wd)] = acc

print("\nTask 3 Results (L2 Regularization):")
for wd, acc in wd_results.items():
    print(f"WD {wd}: Test Acc {acc:.4f}")

### Task 4 (≈20 min): Data Augmentation — **20 / 100 points**

**Goal:** Build a **training** `transform` with **`RandomHorizontalFlip`**, **`RandomRotation`** (e.g. degrees=10), plus `ToTensor` and `Normalize`. Keep **test** transform unchanged. Retrain and report **test accuracy** vs the non-augmented baseline.

**Important:** MNIST digits are nearly centered; augmentation is **mild**. The point is to practice **`transforms`**—on harder datasets, augmentation matters more.

**Deliverable:** Before/after test accuracy + one sentence on improvement or trade-offs.

**Hints:**
- Apply **PIL**-compatible transforms **before** `ToTensor()`; **normalize** after `ToTensor()` using the usual MNIST mean/std from Part 1.
- `torchvision.datasets.MNIST(..., transform=train_transform)` for train; separate `test_transform` for test.
- Compare fairly: same **epochs**, **LR**, **architecture**, **seed** as baseline—only the **train** transform changes.


In [ ]:
# Task 4: Data Augmentation
aug_transform = T.Compose([
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,))
])

train_ds_aug = torchvision.datasets.MNIST(root=data_dir, train=True, download=True, transform=aug_transform)
train_loader_aug = DataLoader(train_ds_aug, batch_size=64, shuffle=True)

set_seed(42)
model_aug = SimpleCNN().to(device)
acc_aug = train_eval(model_aug, train_loader_aug)
print(f"Task 4 Augmentation Test Acc: {acc_aug:.4f}")

<a id="task5"></a>
### Task 5 (≈20 min): Deeper architecture — more convolutional layers — **20 / 100 points**

**Goal:** Extend the **Part 1** `SimpleCNN` by adding **more conv layers** (e.g. **two convolutions per block** before each pool, VGG-style, or an extra block with compatible downsampling). Train with the **same** hyperparameters (epochs, batch size, LR) as the tutorial when possible and compare **test accuracy**, **parameter count**, and **training time** vs `SimpleCNN`.

**Why it matters:** Deeper stacks can learn richer features but may **overfit** small datasets or need **more regularization**—a core trade-off in both **classification** and **generative** CNN backbones.

**Deliverable:** Table or short paragraph comparing baseline vs deeper net + one plot (e.g. test accuracy vs epoch). If the deeper net underperforms, briefly hypothesize why (overfitting, optimization, capacity vs data).

**Hints:**
- Trace tensor shape after **each** conv and pool: MNIST with three `MaxPool2d(2)` typically ends near **3×3** feature maps if you match Part 1’s padding/kernel choices.
- **`nn.Linear` in_features** must equal `channels * height * width` after `flatten`—use a dummy input `torch.zeros(1, 1, 28, 28, device=device)` to verify.
- Count parameters: `sum(p.numel() for p in net.parameters())`.
- Optional: use `%%time` once per training run to compare wall time.


In [ ]:
# Task 5: Deeper Architecture
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

set_seed(42)
m_deep = DeeperCNN().to(device)
params = sum(p.numel() for p in m_deep.parameters())
acc_deep = train_eval(m_deep, train_loader)
print(f"Task 5 Deeper Acc: {acc_deep:.4f} | Total Params: {params}")

### Bonus Task (optional): Harder Dataset — Fashion-MNIST

**Goal:** Swap **MNIST** for **Fashion-MNIST**. We use the same **SimpleCNN** idea but adjust the **normalization** mean and std for the fashion dataset.

**Deliverable:** Test accuracy on 10 fashion categories.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import os

# Ensure environment variables and helpers are available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = os.path.join(os.getcwd(), "data")

def set_seed(seed: int = 42) -> None:
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 1. Define Transforms with Fashion-MNIST normalization
# Fixed: T is now imported as torchvision.transforms above
fashion_transform = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])

# 2. Load Fashion-MNIST
f_train_ds = torchvision.datasets.FashionMNIST(root=data_dir, train=True, download=True, transform=fashion_transform)
f_test_ds = torchvision.datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=fashion_transform)
f_train_loader = DataLoader(f_train_ds, batch_size=64, shuffle=True)
f_test_loader = DataLoader(f_test_ds, batch_size=64, shuffle=False)

# 3. Training & Eval
set_seed(42)
fashion_model = SimpleCNN(num_classes=10).to(device)
opt = torch.optim.Adam(fashion_model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

print("Training on Fashion-MNIST...")
for epoch in range(5):
    fashion_model.train()
    for xb, yb in f_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        crit(fashion_model(xb), yb).backward()
        opt.step()

    fashion_model.eval()
    c, t = 0, 0
    with torch.no_grad():
        for xb, yb in f_test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (fashion_model(xb).argmax(1)==yb).sum().item()
            t += yb.size(0)
    print(f"Epoch {epoch+1} Test Acc: {c/t:.4f}")

print(f"\nFinal Fashion-MNIST Test Accuracy: {c/t:.4f}")

<a id="summary"></a>
## Lab Summary & Takeaways

- **CNNs** stack **convolutions**, **nonlinearities**, and **pooling** to learn **hierarchical visual features** from raw pixels.
- **Training** requires careful choice of **learning rate**, **batch size**, and **optimizer**—these interact and should be validated on a **held-out test** set.
- **Regularization** (**dropout**, **weight decay**) and **data augmentation** reduce **overfitting** and often improve **generalization**.
- **Monitoring** **loss** and **accuracy** curves, plus a **confusion matrix**, makes debugging and comparison systematic.
- **Depth** and **width** increase net **capacity**; match them to **data size** and **regularization**, or you may see **overfitting** or slower optimization.

### Next Steps for Generative AI

The same **convolutional** ideas appear throughout generative modeling:

- **GANs:** convolutional **generator** and **discriminator** (e.g., DCGAN).
- **Diffusion models:** **U-Net**-style CNNs predict noise or score functions (e.g., models behind **Stable Diffusion** use CNN backbones + attention).
- **Autoencoders / VAEs:** CNN **encoders** and **decoders** map images to latent spaces and back.

Mastering **classification CNNs** is a direct stepping stone to reading and modifying these architectures.


<a id="refs"></a>
## References & Further Reading

- [PyTorch Tutorials — Training a Classifier](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [torchvision.datasets](https://pytorch.org/vision/stable/datasets.html)
- [LeCun et al., Gradient-based learning applied to document recognition (1998)](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf) — classic CNN / MNIST.
- [Krizhevsky et al., ImageNet Classification with Deep CNNs (AlexNet, 2012)](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf)
